# Clase 14: Aprendizaje Supervisado 2 - Clasificación


**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**

## Objetivos de la Clase

Conceptual
- Métricas de evaluación de clasificación: Accuracy, Precision, Recall, F1 y ROC-AUC
- Algoritmo de árbol de decisión
- Overfitting, generalización y curvas de aprendizaje
- Data leakage y cómo evitarlo

Aplicación
- Clasificación con `DecisionTreeClassifier`
- Pipeline de clasificación con sklearn
- Holdout y cross-validation
- Feature importance
- AutoML con FLAML

---

<br>

<div align='center'>
<img src="../../recursos/2023-01/18-Aprendizaje-Supervisado-I/machine_learning.png?raw=true" alt="Panorama General ML: Clasificación supervisada, No supervisada y Aprendizaje Reforzado binario" width=700/>
</div>


## Aprendizaje Supervisado

Como vimos en la **Clase 13**, en aprendizaje supervisado cada observación combina un vector de **características** $X$ con una **etiqueta** $Y$. Según el tipo de etiqueta:

- **Categoría/Etiqueta** → **Clasificación** ← *tema de esta clase*
- **Valor real** → **Regresión** ← *clase anterior*

El objetivo es construir modelos capaces de asignar automáticamente clases o valores a nuevas observaciones, en base a lo aprendido de datos etiquetados.

### Framework General de Aprendizaje Supervisado

El mismo *pipeline* que vimos para regresión aplica para clasificación:

1. **Feature Engineering y Preprocesamiento**: preparar los datos.
2. **Entrenar** el algoritmo con los datos.
3. **Evaluar** el rendimiento del modelo generado.
4. **Optimizar** hiperparámetros.

En esta clase nos enfocamos en las métricas de evaluación propias de **clasificación** y en el algoritmo de **árbol de decisión**.

El modelo construido debe **generalizar**, es decir, ser capaz de hacer predicciones correctas en nuevas observaciones. El modelo genera un *decision boundary* que separa las clases; cuanto más holgado, mejor generaliza.

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/overfitting.png?raw=true' width=400/>

</div>

<div align='center'>
<a href='https://www.researchgate.net/figure/Example-of-overfitting-in-classification-a-Decision-boundary-that-best-fits-training_fig1_349186066'>Ejemplo de *Overfitting* en researchgate</a>
</div>

<br/>

### ¿Qué algoritmo utilizar?

Más allá de la **capacidad predictiva**, hay otros factores relevantes:

- **Costo computacional**: tiempo de entrenamiento, latencia en predicción, recursos disponibles.
- **Explicabilidad**: ¿puedo justificar por qué el modelo clasifica de esa manera?
- **Fairness**: ¿el modelo discrimina injustamente a algún grupo?
- **Rendimiento**: si todo lo anterior se cumple, gana el que predice mejor.

### ¿Cómo saber si un modelo es bueno o no?

Lo primero que nos tenemos que preguntar es qué significa ser bueno.

1. Resuelva bien el problema planteado, según la **métrica adecuada**.
2. Cumple con los requisitos de los *stakeholders* involucrados (en cuanto a *fairness*, interpretabilidad, velocidad).
3. **Generaliza bien** a nuevos datos.

Resumimos la capacidad predictiva de un modelo mediante **métricas de desempeño**, las cuales contrastan los valores predichos versus los valores reales de la variable objetivo (siempre con datos no usados durante el entrenamiento).

## Nuestro problema de hoy: Pingüinos  🐧


Origen del dataset:

**Palmer Archipelago (Antarctica) penguin data**: 

*Data were collected and made available by Dr. Kristen Gorman and the Palmer Station, Antarctica LTER, a member of the Long Term Ecological Research Network.*

https://github.com/allisonhorst/palmerpenguins

![Pinguinos](https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/man/figures/lter_penguins.png)


    
    


### Atributos
 
- `culmen_length_mm`: Largo del culmen (vértice o borde superior de la mandíbula)  (mm).
- `culmen_depth_mm`: Alto del culmen (vértice o borde superior de la mandíbula) (mm).
- `flipper_length_mm`: Longitud de las aletas (mm).
- `body_mass_g`: Masa corporal (g).
- `island`: Isla de origen (Dream, Torgersen, or Biscoe) en el archipiélago de Palmer (Antarctica).
- `sex`: Sexo del pinguino.

![Detalle Variables](https://allisonhorst.github.io/palmerpenguins/reference/figures/culmen_depth.png)
    
<center>Créditos a Allison Horst por sus excelentes ilustraciones https://github.com/allisonhorst </center>    
    
    
### Variable a predecir

- `species`: Especie del pinguino (Chinstrap, Adélie, or Gentoo)

### Exploración y Preprocesamiento

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = (
    pd.read_csv("../../recursos/2023-01/18-Aprendizaje-Supervisado-I/penguins.csv")
    .dropna()
    .reset_index(drop=True)
)
df

In [ ]:
import plotly.express as px

fig = px.scatter_matrix(
    df, dimensions=df.iloc[:, 2:].columns, color="species", height=1000
)
fig.show()

> **Pregunta ❓**: Imagina un modelo que predice la clase mayoritaria para **todas** las observaciones. Si el 95 % de los datos pertenece a una sola clase, ¿qué *accuracy* tendría ese modelo? ¿Lo considerarías bueno?

## ⚖️ Métricas de Desempeño en Clasificación

<div align='center'>
    <img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf.PNG?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" width=450/>
</div>

<center>Ejemplo de una matriz de confusión para un problema de clasificación binario.</center>

---

> Ejemplo: Alergia a cierto medicamento en donde la clase `+` indica alergia.


Nuestro dataset tiene 10.000 observaciones distribuidos de la siguiente forma:

- Clase `+`: 100 observaciones.
- Clase `-`: 9900 observaciones.


Luego, creamos un modelo que clasificó nuestro dataset y graficamos sus resultados a través de la siguiente matriz de confusión:


|                    | **Predicha (`+`)**  | **Predicha (`-`)** |
|--------------------|---------------------|--------------------|
| **Real (`+`)**     | 10                  | 90                 |
| **Real (`-`)**     | 100                 | 9800               |

---

> **Pregunta**: ¿Cuáles métricas de desempeño conocen para evaluar este caso?

### Métricas de desempeño

Métricas basadas en contar datos correcta e incorrectamente clasificados:

- **Accuracy (Exactitud)**: $$\text{accuracy} = \frac{\text{número de predicciones correctas}}{\text{número de predicciones totales}}$$

- **Error rate (Tasa de error)**: $$\text{error rate} = \frac{\text{número de predicciones incorrectas}}{\text{número de predicciones totales}}$$



- En nuestro ejemplo anterior: 

$$\text{accuracy} = \frac{9810}{10000} = 0.981$$

$$\text{error rate} = \frac{190}{10000} = 0.019$$


> **Pregunta ❓**: ¿Cuál es el problema de `Accuracy` en nuestro ejemplo?

> **Pregunta ❓**: Un médico que siempre responde "no tiene cáncer" a todos sus pacientes tiene 99% de accuracy si solo el 1% de los pacientes tiene cáncer. ¿Lo contratarías?



---


#### Métricas Basadas en la Matriz de Confusión

Una posible solución a este problema son las métricas basadas en la matriz de confusión:

<div align='center'>
    <img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf.PNG?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" width=450/>
</div>

- **Precision**:  Fracción de ejemplos correctamente predichos como `+` con respecto a todos los predichos `+`.

$$P = \frac{\text{Clasificados correctamente como positivo}}{\text{Todos los predichos como positivos}} =\frac{TP}{(TP + FP)}$$


<br>

- **Recall**: Fracción de ejemplos `+` que son correctamente clasificados sobre el total de ejemplos `+`: 

$$R = \frac{\text{Clasificados correctamente como positivo}}{\text{Todos los que debería haber clasificado como positivos}}  = \frac{TP}{(TP+FN)}$$


<br>

- **F1 measure**: Combina precisión y recall usando una media armónica (i.e., media que castiga si ambos valores son muy diferentes).

$$F = \frac{2PR}{(P+R)}$$



|                    | **Predicha (`+`)**  | **Predicha (`-`)** |
|--------------------|---------------------|--------------------|
| **Real (`+`)**     | 10                  | 90                 |
| **Real (`-`)**     | 100                 | 9800               |

En nuestro ejemplo anterior:


$$
\begin{aligned}
P &= \frac{10}{110} \approx 0.0909 \\
R &= \frac{10}{100} = 0.1 \\
F &= \frac{2 \cdot P \cdot R}{P + R} = \frac{2 \cdot 0.0909 \cdot 0.1}{0.0909 + 0.1} = \frac{2}{21} \approx 0.095
\end{aligned}
$$

Ahora claramente se nota el problema.

> **Pregunta ❓**: En el caso de un problema de detección de frutas podridas, ¿que nos convendría más utilizar? ¿precision o recall? 

![](https://www.lavanguardia.com/files/article_main_microformat/uploads/2019/11/05/5e9979d2bdba1.jpeg)

#### Matriz de confusión multiclase

Cuando tenemos $k$ clases, la matriz de confusión es una matriz de $k \times k$.

<div align='center'>
<img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf_multiclase.png?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" style="width: 500px;"/>
</div>

¿Cómo calculamos las métricas?


#### Métricas de Desempeño Generalizadas: 


- Precision: Fracción de ejemplos asignados a la clase `i` que son realmente de la clase `i`. Finalmente, es la fracción de casos en los que declaramos correctamente 𝑖 de todos los casos en los que el algoritmo declaró 𝑖.

$$\text{precision} = \frac{c_{ii}}{\sum_{j}c_{ji}}$$


- Recall: Fracción de ejemplos de la clase `i` correctamente clasificados. Recall es la fracción de eventos en los que declaramos correctamente 𝑖 de todos los casos en los que el verdadero estado 𝑖: 

$$\text{recall} = \frac{c_{ii}}{\sum_{j}c_{ij}}$$

- Accuracy: Fracción de ejemplos correctamente clasificados:

$$\text{accuracy} = \frac{\sum_{i}c_{ii}}{\sum_{j}\sum_{i}c_{ij}}$$

#### Estrategia de Agregación

- **Macroaveraging**
    - Computar métrica para cada clase y luego promediar. 
    - Sobrerepresentan clases minoritarias al tratar a todas por igual.

- **Weighted**
    - Computar métrica para cada clase y luego hace un promedio ponderado por el número de ejemplos de esa clase.
    - Al ser ponderado por el número de casos, da más prioridad a las clases frecuentes.

- **Microaveraging**
    - Computar métrica sobre los resultados globales, sin discriminar por clase. 
    - Refleja el rendimiento global del modelo, agnóstico a la clases.


`scikit-learn` provee un acceso rápido a todas estas métricas a través de su función `sklearn.metrics.classification_report`

### Curva ROC y AUC

Además de las métricas basadas en la matriz de confusión, una de las más usadas en clasificación es la **ROC-AUC**.

- **TPR (Recall)**: fracción de positivos reales correctamente identificados. $TPR = \frac{TP}{TP + FN}$
- **FPR**: fracción de negativos clasificados erróneamente como positivos. $FPR = \frac{FP}{FP + TN}$

La **curva ROC** grafica TPR vs FPR para todos los umbrales posibles. El área bajo esta curva (**AUC**) resume el rendimiento en un solo número:

- **AUC = 1.0**: clasificador perfecto.
- **AUC = 0.5**: equivale a clasificar aleatoriamente.
- **AUC < 0.5**: peor que aleatorio.

<div align='center'>
<img src='https://upload.wikimedia.org/wikipedia/commons/thumb/1/13/Roc_curve.svg/600px-Roc_curve.svg.png' width=380/>
</div>

**Ventaja clave**: ROC-AUC es invariante al umbral de decisión y robusto a desbalance de clases, lo que la hace muy útil cuando `accuracy` puede ser engañosa.

---

# 🌳 Árboles de Decisión

Un **Árbol de Decisión** es un modelo de aprendizaje supervisado que funciona como un diagrama de flujo. 


### 🏗️ Anatomía de un Árbol

1.  **Nodo Raíz (Root Node):** Representa la primera decisión. Es la variable que mejor divide tus datos.
2.  **Nodos Internos:** Preguntas intermedias que dividen el flujo según los valores de las features.
3.  **Nodos Hoja (Leaf Nodes):** El final del camino. Aquí es donde se asigna la **clase predicha** (en clasificación) o el **valor promedio** (en regresión).


### 🧠 ¿Cómo decide qué preguntar? (Criterios de División)

El árbol es "ambicioso" (*greedy*): en cada paso, busca la división que genere los grupos más **puros** posibles.

#### 1. Entropía (Incertidumbre)

Mide el desorden del sistema. Si un nodo tiene una mezcla 50/50 de clases, la entropía es máxima. El objetivo es reducirla a 0.

$$H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

#### 2. Índice Gini (Impureza)

Es el criterio por defecto en `sklearn`. Mide qué tan seguido un elemento elegido al azar sería clasificado incorrectamente.

$$\text{Gini}(S) = 1 - \sum_{i=1}^{C} p_i^2$$

> 💡 **Ganancia de Información:** Es la diferencia de impureza antes y después de la división. El árbol siempre elige la pregunta que maximiza esta ganancia.


### ⚙️ El Algoritmo (ID3 / CART)

El proceso detrás de cámaras sigue un esquema de **Particionamiento Recursivo**:

1.  **Comenzar** con el conjunto de datos completo en el nodo raíz.
2.  **Evaluar** todas las posibles divisiones para cada feature usando Gini o Entropía.
3.  **Seleccionar** la división con la mayor Ganancia de Información.
4.  **Dividir** los datos en dos subconjuntos (ramas).
5.  **Repetir** el proceso de forma recursiva en cada subconjunto.
6.  **Detenerse** cuando:
    * Todos los datos en un nodo pertenecen a la misma clase.
    * Se alcanza la profundidad máxima definida (`max_depth`).
    * No hay ganancia significativa en seguir dividiendo.


### ⚖️ Fortalezas y Debilidades

| Ventaja | Desventaja |
| :--- | :--- |
| **Caja Blanca:** Muy fácil de interpretar y visualizar. | **Inestabilidad:** Pequeños cambios en los datos pueden cambiar todo el árbol. |
| **No requiere escalado:** Funciona igual con datos en escalas locas. | **Sobreajuste (Overfitting):** Si no se limita, el árbol "memoriza" el ruido. |
| **Versátil:** Maneja datos numéricos y categóricos. | **Bajo rendimiento:** Por sí solos, suelen ser menos precisos que modelos complejos. |


### 🧐 El Reto del Overfitting

> **Pregunta❓:** Si entrenas un árbol sin límites (`max_depth=None`), ¿qué accuracy esperarías en el set de entrenamiento vs. el de test?
>
> * **Entrenamiento:** Tendrás casi un **100% de accuracy**, pues el árbol creará una hoja para cada dato si es necesario.
> * **Test:** Probablemente sea **pobre**. El modelo se volvió tan específico que perdió su capacidad de generalizar (Overfitting).
>
> *¡Por eso el "punning" (poda) y los límites de profundidad son vitales!*

> **Pregunta ❓**: Si entrenas y evalúas un modelo con los **mismos** datos, ¿qué problema tiene esa evaluación? ¿Por qué no podemos confiar en esos resultados?

## ✂️ Holdout

Consiste en particionar nuestro dataset en conjuntos de:

- **Training**: conjunto que se utiliza para **entrenar** el modelo.
- **Testing**: datos que se usa para **evaluar** qué tan bien predice el modelo (a través de las métricas de evaluación).


Comúnmente se dividen en proporción $2/3$ y $1/3$ del dataset respectivamente. Sin embargo, todo depende de la cantidad de datos que se posean: si se tiene millones de ejemplos, quizás puede dividirse en 95% train, 5% test sin problemas.


La evaluación puede variar mucho según las particiones escogidas:

- Training pequeño → modelo sesgado
- Testing pequeño → evaluación poco confiable


Esta técnica se denomina *Random Subsampling* y selecciona aleatoriamente las observaciones de cada conjunto.

Para ejecutar todo esto usaremos `train_test_split`. Veamos algunos de sus parámetros:

- `test_size = 0.33` - indica el tamaño del conjunto de evaluación.
- `shuffle = True` - indica que ejecutaremos Random Subsampling.
- `stratify = labels` - intenta mantener la distribución de clases original en ambos conjuntos.

### Conjunto de Validación

Cuando se desea realizar una búsqueda de los mejores algoritmos y sus hiperparámetros, el dataset puede ser dividido en 3:


- **Training**: Se utiliza para entrenar los modelos.
- **Validation**: Se utiliza para seleccionar el mejor modelo al ir variando sus hiperparámetros.
- **Testing**: Se utiliza para evaluar el modelo previo a ser entregado o puesto en producción. Esta evaluación solo se hace sobre el modelo final.


En este caso la división puede ser $70\%, 15\%, 15\%$ respectivamente.

In [ ]:
# Holdout
from sklearn.model_selection import train_test_split

features = df.drop(columns=["species"])
labels = df.loc[:, "species"]


X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.33, shuffle=True, stratify=labels, random_state=42
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# conjunto de entrenamiento
y_train.value_counts() / y_train.count() * 100

In [ ]:
# conjunto de pruebas
y_test.value_counts() / y_test.count() * 100

Entrenemos ahora nuestro primer modelo de clasificación con el dataset de pingüinos usando un árbol de decisión dentro de un Pipeline:

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

ct = ColumnTransformer(
    [
        (
            "Scaler",
            RobustScaler(),
            [
                "culmen_length_mm",
                "culmen_depth_mm",
                "flipper_length_mm",
                "body_mass_g",
            ],
        ),
        (
            "OneHot",
            OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist"),
            ["island", "sex"],
        ),
    ]
)

tree_pipe = Pipeline(
    [
        ("preprocesamiento", ct),
        ("tree", DecisionTreeClassifier(criterion="entropy")),
    ]
)

# El preprocesamiento está dentro del pipeline → entrenamos con X sin transformar
tree_pipe.fit(X_train, y_train)

In [ ]:
y_pred = tree_pipe.predict(X_test)
len(y_pred)

### Evaluación

In [ ]:
print("Matriz de confusión\n\n", confusion_matrix(y_test, y_pred), "\n")
print(classification_report(y_test, y_pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)
labels = sorted(y_test.unique())

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    ax=ax,
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusión")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

# Para multiclase se entregan probabilidades con predict_proba
y_prob = tree_pipe.predict_proba(X_test)

# One-vs-Rest, promedio macro
auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro")
print(f"ROC-AUC (macro, OVR): {auc:.3f}")

### Curva ROC por clase (One-vs-Rest)

El número de AUC que calculamos arriba resume el rendimiento en un escalar, pero la **curva ROC** muestra el trade-off TPR/FPR para cada umbral y para cada clase individualmente, lo cual es mucho más informativo.

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc
import matplotlib.pyplot as plt

classes = tree_pipe.classes_
y_prob = tree_pipe.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ["steelblue", "darkorange", "green"]

for idx_c, (cls, color) in enumerate(zip(classes, colors)):
    y_bin = (y_test == cls).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_prob[:, idx_c])
    roc_auc = sklearn_auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{cls} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Aleatorio (AUC = 0.5)")
ax.set_xlabel("Tasa de Falsos Positivos (FPR)")
ax.set_ylabel("Tasa de Verdaderos Positivos (TPR / Recall)")
ax.set_title("Curva ROC — One-vs-Rest por clase")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Widget: Umbral de decisión → Precision / Recall

Los clasificadores producen **probabilidades**, y la clase final se asigna comparando esa probabilidad contra un **umbral** (por defecto 0.5). Cambiar el umbral desplaza el balance entre Precision y Recall:

- **Umbral bajo** → el modelo clasifica más observaciones como positivas → sube el Recall pero baja la Precision.
- **Umbral alto** → el modelo solo predice positivo cuando está muy seguro → sube la Precision pero baja el Recall.

Usa el slider para ver este efecto en vivo para cada especie (One-vs-Rest):

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

_classes = tree_pipe.classes_
_y_prob = tree_pipe.predict_proba(X_test)

_out_threshold = widgets.Output()


def _show_threshold(clase, umbral):
    idx_c = list(_classes).index(clase)
    y_bin_r = (y_test == clase).astype(int)
    y_bin_p = (_y_prob[:, idx_c] >= umbral).astype(int)

    cm = confusion_matrix(y_bin_r, y_bin_p)
    prec = precision_score(y_bin_r, y_bin_p, zero_division=0)
    rec = recall_score(y_bin_r, y_bin_p, zero_division=0)
    f1 = f1_score(y_bin_r, y_bin_p, zero_division=0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=axes[0],
        xticklabels=[f"No {clase}", clase],
        yticklabels=[f"No {clase}", clase],
    )
    axes[0].set_xlabel("Predicho")
    axes[0].set_ylabel("Real")
    axes[0].set_title(f"Matriz de Confusión  (umbral = {umbral:.2f})")

    bars = axes[1].bar(
        ["Precision", "Recall", "F1"],
        [prec, rec, f1],
        color=["steelblue", "darkorange", "green"],
    )
    axes[1].set_ylim(0, 1.15)
    axes[1].set_title(f"{clase} vs. resto")
    for bar, val in zip(bars, [prec, rec, f1]):
        axes[1].text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.03,
            f"{val:.2f}",
            ha="center",
            fontsize=11,
        )

    plt.tight_layout()
    with _out_threshold:
        _out_threshold.clear_output(wait=True)
        plt.show()
    plt.close(fig)


ui = widgets.interactive(
    _show_threshold,
    clase=widgets.Dropdown(
        options=list(_classes),
        value=_classes[1],
        description="Clase:",
        style={"description_width": "initial"},
    ),
    umbral=widgets.FloatSlider(
        min=0.01,
        max=0.99,
        step=0.01,
        value=0.5,
        description="Umbral:",
        readout_format=".2f",
        style={"description_width": "initial"},
    ),
)

display(ui, _out_threshold)


> **Pregunta ❓**: ¿Qué ventaja tiene un modelo cuya lógica puedes leer como un diagrama de flujo, comparado con uno que actúa como una "caja negra"?

---

## 🌳 Visualización del Árbol de Decisión

Una de las grandes ventajas de los árboles de decisión es su **interpretabilidad**: podemos graficar exactamente la lógica que aprendió el modelo.

A continuación visualizamos el árbol entrenado. Para que sea legible, mostramos solo los primeros **3 niveles de profundidad** (`max_depth=3`). El árbol real puede ser más profundo.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Extraemos el árbol y los nombres de features/clases del pipeline
tree_model = tree_pipe["tree"]
feature_names = tree_pipe["preprocesamiento"].get_feature_names_out().tolist()
class_names = tree_model.classes_.tolist()

fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,  # colorea los nodos según la clase dominante
    rounded=True,  # bordes redondeados
    impurity=True,  # muestra entropía/gini en cada nodo
    proportion=False,  # muestra conteos absolutos (no porcentajes)
    max_depth=3,  # limita la visualización a 3 niveles
    fontsize=9,
    ax=ax,
)
ax.set_title("Árbol de Decisión entrenado — primeros 3 niveles", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nProfundidad total del árbol: {tree_model.get_depth()}")
print(f"Número total de hojas     : {tree_model.get_n_leaves()}")

### 📖 Cómo leer el árbol

Cada **nodo** (rectángulo) contiene la siguiente información:

| Campo | Significado |
|---|---|
| `flipper_length_mm <= 0.6` | **Condición de división**: si se cumple, va por la rama izquierda; si no, por la derecha. El valor está en escala transformada (RobustScaler). |
| `entropy = 0.83` | **Impureza** del nodo: qué tan mezcladas están las clases. `0.0` = nodo puro; `log₂(k)` = máxima mezcla para *k* clases. |
| `samples = 223` | **Número de ejemplos** de entrenamiento que llegan a ese nodo. |
| `value = [99, 45, 79]` | **Conteo por clase** (orden: `Adelie, Chinstrap, Gentoo`). |
| `class = Adelie` | **Clase predicha** si se detuviera aquí: la clase con mayor conteo. |

#### 🎨 Color del nodo

El color indica la **clase dominante** y su intensidad refleja la **pureza**:
- **Más saturado** → nodo más puro (una clase claramente mayoría).
- **Más blanco/pastel** → clases muy mezcladas.

#### ⚠️ Señales de overfitting en el árbol

- Árbol muy **profundo** con hojas de `samples = 1`.
- Nodos hoja con `entropy ≈ 0` pero `samples` muy pequeño: el modelo memorizó en vez de aprender.
- Gran diferencia entre accuracy en training vs. test (ver sección siguiente).

> **Solución**: limitar `max_depth`, `min_samples_leaf` o `min_samples_split`.

> **Pregunta ❓**: ¿Cuándo podemos decir que un modelo está sobre-ajustado?

### Demostración de Overfitting

Entrenemos árboles con distintas profundidades y comparemos la accuracy en **entrenamiento** vs. **test**. Un árbol muy profundo memoriza el set de entrenamiento (accuracy ≈ 100 %) pero generaliza mal.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

depths = list(range(1, 16))
train_accs, test_accs = [], []

_num = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
_cat = ["island", "sex"]

for d in depths:
    _ct = ColumnTransformer(
        [
            ("scaler", RobustScaler(), _num),
            (
                "ohe",
                OneHotEncoder(
                    sparse_output=False, handle_unknown="infrequent_if_exist"
                ),
                _cat,
            ),
        ]
    )
    _pipe = Pipeline(
        [("prep", _ct), ("tree", DecisionTreeClassifier(max_depth=d, random_state=42))]
    )
    _pipe.fit(X_train, y_train)
    train_accs.append(_pipe.score(X_train, y_train))
    test_accs.append(_pipe.score(X_test, y_test))

best_depth = depths[test_accs.index(max(test_accs))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_accs, "o-", label="Train accuracy", color="steelblue")
ax.plot(depths, test_accs, "o-", label="Test accuracy", color="darkorange")
ax.axvline(
    x=best_depth,
    color="gray",
    linestyle="--",
    alpha=0.7,
    label=f"mejor depth={best_depth}",
)
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.set_title("Overfitting en árboles de decisión")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mejor max_depth según test accuracy: {best_depth}")

### Widget: Decision Boundary

Una forma muy intuitiva de entender el overfitting es ver cómo cambia la **frontera de decisión** al aumentar `max_depth`. Con profundidad baja el modelo traza fronteras simples y generalizables; con profundidad alta fragmenta el espacio en regiones muy pequeñas que memorizan el ruido.

Selecciona dos features y ajusta la profundidad para ver el efecto:

In [ ]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

_feat_opts = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
_species_colors = ["steelblue", "darkorange", "green"]


def _show_boundary(max_depth, feat_x, feat_y):
    if feat_x == feat_y:
        print("Elige dos features distintas.")
        return

    X2 = df[[feat_x, feat_y]].values
    y2 = df["species"].values

    clf2 = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    clf2.fit(X2, y2)

    pad = 0.5
    x_min, x_max = X2[:, 0].min() - pad, X2[:, 0].max() + pad
    y_min, y_max = X2[:, 1].min() - pad, X2[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()])
    Z_num = np.array([list(clf2.classes_).index(z) for z in Z]).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(
        xx, yy, Z_num, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], colors=_species_colors
    )
    ax.contour(
        xx, yy, Z_num, levels=[0.5, 1.5], colors="gray", linewidths=0.8, linestyles="--"
    )

    for j, cls in enumerate(clf2.classes_):
        mask = y2 == cls
        ax.scatter(
            X2[mask, 0],
            X2[mask, 1],
            label=cls,
            color=_species_colors[j],
            edgecolors="k",
            linewidths=0.4,
            s=40,
            zorder=3,
        )

    train_acc = clf2.score(X2, y2)
    ax.set_xlabel(feat_x)
    ax.set_ylabel(feat_y)
    ax.set_title(
        f"Decision boundary — max_depth={max_depth}   train acc={train_acc:.2f}"
    )
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


widgets.interact(
    _show_boundary,
    max_depth=widgets.IntSlider(
        min=1,
        max=12,
        step=1,
        value=3,
        description="max_depth:",
        style={"description_width": "initial"},
    ),
    feat_x=widgets.Dropdown(
        options=_feat_opts,
        value="flipper_length_mm",
        description="Eje X:",
        style={"description_width": "initial"},
    ),
    feat_y=widgets.Dropdown(
        options=_feat_opts,
        value="culmen_depth_mm",
        description="Eje Y:",
        style={"description_width": "initial"},
    ),
);

> **Pregunta ❓**: Si tienes solo 100 muestras y reservas el 20 % para test, entrenas con apenas 80 ejemplos. ¿Existe alguna forma de usar **todos** los datos tanto para entrenar como para evaluar, sin mezclarlos?

## 🔀 Cross-Validation

*Cross-validation* es una técnica para evaluar el grado de generalización de un modelo en un conjunto de datos independiente. Es especialmente útil para conjuntos de datos pequeños, en los que cada punto de datos es valioso tanto para el entrenamiento como para la validación. El proceso consiste en dividir el conjunto de datos en varias partes, utilizar algunas para entrenar el modelo y el resto para probarlo, y repetir este proceso varias veces para garantizar una evaluación completa.

Aunque normalmente se utiliza para la selección de modelos y la estimación del rendimiento, la validación cruzada también puede proporcionar información sobre la estabilidad y la incertidumbre de las predicciones de su modelo. Al examinar la variación de las métricas de rendimiento (p. ej., precisión, RMSE), puede evaluar la coherencia del rendimiento de los modelos en diferentes subconjuntos de datos.

---
    Para cada partición i:
        - Juntar todas las k-1 particiones restantes y entrenar el modelo sobre esos datos.
        - Evaluar el modelo en la partición i.
        
    El error total = suma de errores de todos los modelos  

---

<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/kfold.png?raw=true' width=400>

### La forma rápida: `cross_val_score`

Sklearn ofrece una API de una línea que hace exactamente lo del loop de abajo: itera sobre los folds, re-entrena el pipeline completo en cada uno y devuelve los scores. Como el pipeline incluye el preprocesamiento, **no hay data leakage**.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv_quick = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

acc_cv = cross_val_score(tree_pipe, X_train, y_train, cv=cv_quick, scoring="accuracy")
f1_cv = cross_val_score(tree_pipe, X_train, y_train, cv=cv_quick, scoring="f1_macro")

print(f"Accuracy — media: {acc_cv.mean():.3f}  std: {acc_cv.std():.3f}")
print(f"F1 macro — media: {f1_cv.mean():.3f}  std: {f1_cv.std():.3f}")

### Implementación manual (para entender qué hace `cross_val_score` por dentro)

A veces necesitamos más control: guardar predicciones por fold, calcular métricas personalizadas, etc. En ese caso recorremos los folds explícitamente:

<center>
<img src='https://amueller.github.io/aml/_images/stratified_cv.png' width=300 />


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
import pandas as pd

cv = StratifiedKFold(n_splits=7)
scores = {"acc": [], "f1": []}

for train_index, test_index in cv.split(X_train, y_train):
    X_train_fold = X_train.iloc[train_index]
    X_test_fold = X_train.iloc[test_index]
    y_train_fold = y_train.iloc[train_index]
    y_test_fold = y_train.iloc[test_index]

    # El ColumnTransformer se re-crea y fitea en cada fold → sin data leakage
    ct_fold = ColumnTransformer(
        [
            (
                "Scaler",
                RobustScaler(),
                [
                    "culmen_length_mm",
                    "culmen_depth_mm",
                    "flipper_length_mm",
                    "body_mass_g",
                ],
            ),
            (
                "OneHot",
                OneHotEncoder(
                    sparse_output=False, handle_unknown="infrequent_if_exist"
                ),
                ["island", "sex"],
            ),
        ]
    )

    fold_pipe = Pipeline(
        [
            ("preprocesamiento", ct_fold),
            ("tree", DecisionTreeClassifier(random_state=42)),
        ]
    )
    fold_pipe.fit(X_train_fold, y_train_fold)

    y_pred_fold = fold_pipe.predict(X_test_fold)
    scores["acc"].append(accuracy_score(y_test_fold, y_pred_fold))
    scores["f1"].append(f1_score(y_test_fold, y_pred_fold, average="macro"))

In [ ]:
# Hagamos una tabla delos resultados
results = pd.DataFrame.from_dict(scores)
mean_col = pd.DataFrame(results.mean()).T.rename(index={0: "mean"})
std_col = pd.DataFrame(results.std()).T.rename(index={0: "std"})
results = pd.concat([results, mean_col], axis=0)
results = pd.concat([results, std_col], axis=0)
results

> **Pregunta ❓**: ¿Por qué usamos StratifiedKFold en lugar de KFold simple? ¿Qué problema resuelve cuando las clases están desbalanceadas?



> **Pregunta ❓**: ¿Qué podría pasar si nuestro modelo tiene una alta variación?

> **Pregunta ❓**: ¿Podría existir algún problema al aplicar *feature engineering* sobre el conjunto de datos completo, antes de dividirlo en *train* y *test*?

> **Pregunta ❓**: Si escalas tus *features* usando la media y desviación estándar de **todo** el dataset (train + test) antes de dividirlo, ¿qué información del conjunto de test estaría filtrándose hacia el entrenamiento?

El siguiente gráfico ilustra el efecto:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature = "culmen_length_mm"
X = df[[feature]].values
y = df["species"].values

X_train_raw, X_test_raw, _, _ = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)

# ✅ Correcto: ajustar el scaler SOLO en train
sc_correct = StandardScaler()
X_train_ok = sc_correct.fit_transform(X_train_raw).ravel()
X_test_ok = sc_correct.transform(X_test_raw).ravel()

# ❌ Con Leakage: ajustar el scaler en TODOS los datos (train + test)
sc_leaky = StandardScaler()
sc_leaky.fit(X)  # "ve" los datos de test — ¡esto es incorrecto!
X_train_lk = sc_leaky.transform(X_train_raw).ravel()
X_test_lk = sc_leaky.transform(X_test_raw).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
bins = np.linspace(-3, 3, 30)

for ax, (X_tr, X_te, title) in zip(
    axes,
    [
        (X_train_ok, X_test_ok, "✅ Sin Leakage\n(Scaler ajustado solo en Train)"),
        (X_train_lk, X_test_lk, "❌ Con Leakage\n(Scaler ajustado en Train + Test)"),
    ],
):
    ax.hist(
        X_tr,
        bins=bins,
        alpha=0.65,
        label=f"Train  (μ={X_tr.mean():.3f})",
        color="steelblue",
    )
    ax.hist(
        X_te,
        bins=bins,
        alpha=0.65,
        label=f"Test   (μ={X_te.mean():.3f})",
        color="coral",
    )
    ax.axvline(0, color="k", linestyle="--", linewidth=1, label="μ=0 referencia")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f"{feature} escalado")
    ax.legend(fontsize=9)

plt.suptitle(
    "Efecto del Data Leakage en el Escalado de Features",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

print(
    f"Sin leakage — μ train: {X_train_ok.mean():.4f},  μ test: {X_test_ok.mean():.4f}"
)
print(
    f"Con leakage — μ train: {X_train_lk.mean():.4f},  μ test: {X_test_lk.mean():.4f}"
)
print()
print(
    f"Scaler correcto: μ aprendida = {sc_correct.mean_[0]:.4f}  (calculada solo sobre train)"
)
print(
    f"Scaler leaky:    μ aprendida = {sc_leaky.mean_[0]:.4f}  (calculada sobre train + test)"
)


### 🔍 Preprocesamiento y *Data Leakage*

Uno de los errores más comunes al construir modelos es contaminar el entrenamiento con información del conjunto de prueba. A continuación se explica cuándo ocurre y cómo evitarlo.

<center>
<img src='https://media1.giphy.com/media/lVBtp4SRW6rvDHf1b6/200w.gif?cid=6c09b952a7zf3h2d55w0f4w0qwqam42qbtojlgcpitpq4c8o&ep=v1_gifs_search&rid=200w.gif&ct=g' width=300 />

*Data Leakage* (fuga de datos) se refiere al uso de datos de prueba dentro del entrenamiento de un modelo predictivo (lo que obviamente es incorrecto).

Dentro de los motivos por los cuales se podría generar data leakage, tenemos:

- **Fuga temporal**: Se refiere a dividir datos que están correlacionados en el tiempo de manera aleatoria en lugar de hacerlo de forma cronológica. Esto puede causar problemas en la interpretación y evaluación del modelo.
  
- **Procesamiento de datos antes de la división**: Se refiere a la manipulación de los datos antes de dividirlos en conjuntos de entrenamiento y prueba. Es importante realizar cualquier procesamiento de datos, como normalización o codificación, después de dividir los datos para evitar fugas de información.
  
- **Manejo deficiente de la duplicación de datos antes de la división**: Indica que no se está manejando adecuadamente la duplicación de datos antes de dividirlos en conjuntos de entrenamiento y prueba. Esto puede introducir sesgos en el modelo y afectar su rendimiento.
  
- **Fuga por variables altamente correlacionadas con el target**: Ocurre cuando usamos columnas que contienen información derivada o transformada del target, aunque indirectamente, en nuestro entrenamiento.

Es muy importante que el **preprocesamiento y feature engineering lo hagan siempre sobre los datos de entrenamiento y no sobre todo el dataset**. De lo contrario, estarían ocupando datos destinados a evaluar para entrenar el modelo (o el preprocesamiento) lo que puede inducir a resultados muy buenos cuando en verdad no deberían serlos.

Más información en [data-leakage de scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

In [ ]:
X_train_preprocessed = pd.DataFrame(
    ct.fit_transform(X_train),
    columns=np.concatenate(
        [ct.transformers_[0][2], ct.transformers_[1][1].get_feature_names_out()], axis=0
    ),
)

#### ¿Dónde puede ocurrir *Data Leakage*?

1. **Medir la correlación de una features con las etiquetas**. Revisar en profundidad las correlaciones altas entre feature porque podrían contener información de nuestra etiqueta.

2. **Estudio de ablación de features**. Si la eliminación de una feature provoca una disminución significativa en el rendimiento del modelo, es importante averiguar por qué.

3. **Cómo identificarlo**. Métricas demasiado buenas para ser verdad. Gran brecha entre validación y test real.

> **Pregunta ❓**: ¿Cómo podrías saber si tu modelo mejoraría añadiendo más datos de entrenamiento, o si el problema es que el modelo es demasiado simple para capturar la complejidad del problema?

### 📉 Curvas de Aprendizaje

Una Learning Curve muestra la relación entre la puntuación de entrenamiento y la puntuación de prueba validada cruzadamente para un estimador con un número variable de muestras de entrenamiento. Esta visualización se utiliza típicamente para demostrar:

1. Cuánto se beneficia el estimador a medida que posee más datos.

2. Evaluar si el modelo está sufriendo *underfitting* u *overfitting*.

In [ ]:
import numpy as np
from yellowbrick.model_selection import LearningCurve
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"

cv = StratifiedKFold(n_splits=12)
sizes = np.linspace(0.3, 1.0, 10)

visualizer = LearningCurve(
    tree_pipe, cv=cv, scoring="f1_weighted", train_sizes=sizes, n_jobs=4
)

visualizer.fit(X_train, y_train)
visualizer.show()

En síntesis, esto nos ayuda a:

1. **Optimización del tamaño del conjunto de datos:** Ayuda a determinar si el modelo se beneficiaría de más datos de entrenamiento o si ya ha alcanzado su límite de rendimiento con la cantidad actual de datos. Esto es crucial para evitar el sobreajuste o el subajuste del modelo.

2. **Diagnóstico de problemas de rendimiento:** Permite identificar si el modelo sufre principalmente de sesgo (*underfitting*) o de varianza (*overfitting*). Si la curva de aprendizaje indica un bajo rendimiento tanto en los datos de entrenamiento como en los de validación, es posible que el modelo necesite una mayor complejidad. Si la brecha entre las curvas de aprendizaje es grande, puede ser indicativo de sobreajuste.

3. **Toma de decisiones informadas:** Proporciona información valiosa para tomar decisiones sobre la arquitectura del modelo, la recopilación de datos adicionales o la necesidad de técnicas de regularización para mejorar el rendimiento del modelo.

### 🏆 Importancia de Features

> **Pregunta ❓**: ¿Qué *features* crees que tienen mayor impacto en predecir la especie de un pingüino? ¿Cómo lo verificarías?

La *feature importance* en *machine learning* es una medida que indica la contribución relativa de cada característica (o variable) en la predicción realizada por un modelo. En esencia, proporciona información sobre qué características son más influyentes para el modelo a la hora de hacer predicciones.

Esta medida es útil para entender qué variables están contribuyendo significativamente a la capacidad predictiva del modelo y puede ayudar en tareas como la selección de características, la interpretación del modelo y la identificación de características relevantes para un problema en particular.

Existen diferentes métodos para calcular la importancia de características, dependiendo del tipo de modelo utilizado. Algunos modelos proporcionan directamente esta información (como los árboles de decisión), mientras que en otros casos se pueden aplicar técnicas específicas, como la importancia de características basada en permutaciones o el uso de coeficientes en modelos lineales.

En scikit-learn los árboles calculan la importancia de cada feature como la **reducción total de impureza** (Gini o entropía) que esa variable provoca en todos los nodos del árbol, ponderada por el número de muestras que pasan por cada nodo:

$$\text{importance}(f) = \sum_{t \in \text{nodos que usan } f} \frac{n_t}{N} \cdot \Delta\text{impureza}(t)$$

donde $n_t$ es el número de muestras en el nodo $t$ y $N$ el total. Los valores se normalizan para que sumen 1.

> ⚠️ Esta medida puede sobreestimar la importancia de features con muchos valores únicos. Para una estimación más robusta considera `sklearn.inspection.permutation_importance`.

Veamos la *feature importance* en nuestro árbol:

In [ ]:
columns_names = tree_pipe["preprocesamiento"].get_feature_names_out()
feat_importances = pd.DataFrame(
    tree_pipe["tree"].feature_importances_, index=columns_names, columns=["Importance"]
)
feat_importances.sort_values(by="Importance", ascending=True, inplace=True)
feat_importances.plot(kind="barh", figsize=(8, 6), legend=False)
plt.show()

> **Pregunta ❓**: ¿Cómo podríamos usar la importancia de *features* para reducir la complejidad del modelo sin sacrificar rendimiento?

In [ ]:
import plotly.graph_objects as go
from sklearn.model_selection import cross_val_score

# Preprocesamos X_train una sola vez para poder explorar subconjuntos de features
array_X = tree_pipe["preprocesamiento"].fit_transform(X_train)
cols_trans_name = tree_pipe["preprocesamiento"].get_feature_names_out()
new_X_train = pd.DataFrame(array_X, columns=cols_trans_name)

clf = tree_pipe["tree"]
precision_scores = []

for i in range(1, len(cols_trans_name) + 1):
    selected_features = cols_trans_name[:i]
    scores = cross_val_score(
        clf, new_X_train[selected_features], y_train, cv=5, scoring="precision_macro"
    )
    precision_scores.append(scores)

x = list(range(1, len(cols_trans_name) + 1))
y_mean = np.array(precision_scores).mean(axis=1)
y_std = np.array(precision_scores).std(axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y_mean, mode="lines", name="Precision"))
fig.add_trace(
    go.Scatter(
        x=x + x[::-1],
        y=list(y_mean - y_std) + list((y_mean + y_std)[::-1]),
        fill="toself",
        fillcolor="rgba(0,100,80,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="±1 std",
    )
)
fig.update_layout(
    title="Precisión del modelo según número de features (top-k por importancia)",
    template="simple_white",
    xaxis_title="Número de features",
    yaxis_title="Precision (macro, CV=5)",
)
fig.show()

> **Pregunta ❓**: Tu árbol de decisión logra un 99 % de *accuracy*, pero no puedes explicarle a tu cliente por qué tomó una decisión específica. ¿Lo pondrías en producción?

## Selección de Modelos

Elegir un modelo no es solo optimizar una métrica. Considera siempre estos factores:

| Factor | Qué evaluar |
|---|---|
| **Costo computacional** | Tiempo de entrenamiento, latencia en predicción, recursos disponibles |
| **Explicabilidad** | ¿Puedes justificar la predicción ante un *stakeholder*? |
| **Fairness** | ¿El modelo discrimina injustamente a algún grupo? |
| **Mantenibilidad** | ¿Qué tan fácil es actualizar y depurar el modelo en producción? |
| **Rendimiento** | Si todo lo anterior se cumple, gana el que predice mejor |

### Reglas prácticas

- **Empieza con el modelo más simple** que resuelva el problema. Permite validar el pipeline end-to-end rápido y sirve como *baseline* sólido.
- **No persigas el SOTA** a menos que lo necesites: los modelos de frontera son costosos, difíciles de explicar y con poco soporte en producción.
- **Evalúa los compromisos (*trade-offs*)**: más precisión no siempre justifica mayor opacidad, especialmente en dominios de alto impacto (salud, crédito, justicia).
- **Entiende las suposiciones** de tu modelo antes de confiar en sus predicciones.
- **Evita sesgo humano**: elige modelos basándote en métricas, no en preferencias personales.

### Guía de Scikit-learn para Elegir el Modelo

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/ml_map.png?raw=true' />
</div>

> **Pregunta ❓**: Si quisieras clasificar un pingüino nuevo basándote únicamente en los pingüinos más similares a él que ya están en el dataset, ¿cómo describirías ese algoritmo?

---

# 🔵 K-Nearest Neighbors (KNN)

**K-Nearest Neighbors** es uno de los algoritmos de clasificación más intuitivos: para predecir la clase de un punto nuevo, busca los **k vecinos más cercanos** en el espacio de features y vota por la clase mayoritaria.

### Anatomía

| Componente | Descripción |
|---|---|
| **k** | Número de vecinos a consultar. Hiperparámetro central |
| **Métrica de distancia** | Euclidiana por defecto; también Manhattan, Minkowski |
| **Votación** | La clase más frecuente entre los k vecinos gana |

### ¿Cómo aprende?

No aprende parámetros durante el entrenamiento: simplemente **memoriza los datos** (*lazy learner*). El costo computacional se paga en la predicción, no en el ajuste.

### Efecto de k sobre el bias-variance tradeoff

| k pequeño | k grande |
|---|---|
| Frontera muy flexible → **overfitting** | Frontera suave → **underfitting** |
| Alta varianza, bajo sesgo | Bajo varianza, alto sesgo |

> ⚠️ **Requiere escalado**: las distancias dependen directamente de la escala de las features. Sin `StandardScaler` o `RobustScaler`, features con rangos grandes dominarán el cálculo de distancia.

### Fortalezas y debilidades

| Ventaja | Desventaja |
|---|---|
| Sin suposiciones sobre la distribución de los datos | Lento en predicción para datasets grandes |
| Captura fronteras no lineales | Sensible a features irrelevantes y ruido |
| Fácil de implementar | Requiere escalar las features |


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

_num_cols = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
_cat_cols = ["island", "sex"]

ct_knn = ColumnTransformer(
    [
        ("scaler", RobustScaler(), _num_cols),
        (
            "ohe",
            OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist"),
            _cat_cols,
        ),
    ]
)

knn_pipe = Pipeline(
    [
        ("preprocesamiento", ct_knn),
        ("knn", KNeighborsClassifier(n_neighbors=5)),
    ]
)

knn_pipe.fit(X_train, y_train)
y_pred_knn = knn_pipe.predict(X_test)

print(classification_report(y_test, y_pred_knn))

cm_knn = confusion_matrix(y_test, y_pred_knn, labels=knn_pipe.classes_)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=knn_pipe.classes_,
    yticklabels=knn_pipe.classes_,
    ax=ax,
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("KNN — Matriz de Confusión (k=5)")
plt.tight_layout()
plt.show()

### Widget: k y Decision Boundary

Observa cómo cambia la frontera de decisión al variar **k**:

- **k=1**: cada punto entrena una región exclusiva → frontera muy irregular → overfitting.
- **k grande**: la mayoría vota → frontera suave → underfitting.

El punto óptimo está en el medio.

In [ ]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

_feat_opts = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
_colors = ["steelblue", "darkorange", "green"]


def _show_knn_boundary(k, feat_x, feat_y):
    if feat_x == feat_y:
        print("Elige dos features distintas.")
        return

    X2 = df[[feat_x, feat_y]].values
    y2 = df["species"].values

    # KNN sobre datos crudos (sin escalar) para visualización simple
    from sklearn.preprocessing import RobustScaler

    scaler2 = RobustScaler()
    X2_sc = scaler2.fit_transform(X2)

    clf_knn = KNeighborsClassifier(n_neighbors=k)
    clf_knn.fit(X2_sc, y2)

    pad = 0.5
    x_min, x_max = X2[:, 0].min() - pad, X2[:, 0].max() + pad
    y_min, y_max = X2[:, 1].min() - pad, X2[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid_sc = scaler2.transform(np.c_[xx.ravel(), yy.ravel()])

    Z = clf_knn.predict(grid_sc)
    Z_num = np.array([list(clf_knn.classes_).index(z) for z in Z]).reshape(xx.shape)

    train_acc = clf_knn.score(X2_sc, y2)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z_num, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], colors=_colors)
    ax.contour(
        xx, yy, Z_num, levels=[0.5, 1.5], colors="gray", linewidths=0.8, linestyles="--"
    )
    for j, cls in enumerate(clf_knn.classes_):
        mask = y2 == cls
        ax.scatter(
            X2[mask, 0],
            X2[mask, 1],
            label=cls,
            color=_colors[j],
            edgecolors="k",
            linewidths=0.4,
            s=40,
            zorder=3,
        )
    ax.set_xlabel(feat_x)
    ax.set_ylabel(feat_y)
    ax.set_title(f"KNN — k={k}   train acc={train_acc:.2f}")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


widgets.interact(
    _show_knn_boundary,
    k=widgets.IntSlider(
        min=1,
        max=31,
        step=2,
        value=5,
        description="k (vecinos):",
        style={"description_width": "initial"},
    ),
    feat_x=widgets.Dropdown(
        options=_feat_opts,
        value="flipper_length_mm",
        description="Eje X:",
        style={"description_width": "initial"},
    ),
    feat_y=widgets.Dropdown(
        options=_feat_opts,
        value="culmen_depth_mm",
        description="Eje Y:",
        style={"description_width": "initial"},
    ),
);

> **Pregunta ❓**: Mueve el slider de k de 1 a 31. ¿En qué rango de k el modelo parece generalizar mejor? ¿Qué ocurre con la frontera cuando k=1?

---

# 📈 Regresión Logística

A pesar del nombre, la **Regresión Logística** es un clasificador. Calcula una combinación lineal de las features y la convierte en probabilidad mediante la **función sigmoide** (o softmax en multiclase):

$$P(y=k \mid x) = \frac{e^{w_k \cdot x}}{\sum_j e^{w_j \cdot x}}$$

### Características principales

| Aspecto | Descripción |
|---|---|
| **Frontera de decisión** | Siempre es un **hiperplano** (lineal) |
| **Salida** | Probabilidades calibradas para cada clase |
| **Regularización C** | Inverso de la fuerza de regularización: C alto = poca regularización |
| **Penalty L1 / L2** | L1 → coeficientes dispersos (algunos = 0); L2 → suaviza todos |
| **Requiere escalado** | Sí, los coeficientes dependen de la escala de las features |

### Efecto de C

$$\text{C pequeño} \Rightarrow \text{más regularización} \Rightarrow \text{frontera más simple} \Rightarrow \text{underfitting}$$
$$\text{C grande} \Rightarrow \text{poca regularización} \Rightarrow \text{frontera más ajustada} \Rightarrow \text{riesgo de overfitting}$$

### Fortalezas y debilidades

| Ventaja | Desventaja |
|---|---|
| Muy rápida de entrenar y predecir | Solo captura relaciones **lineales** |
| Probabilidades bien calibradas | Puede fallar si las clases no son separables linealmente |
| Coeficientes interpretables | Requiere feature engineering para relaciones complejas |


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

ct_lr = ColumnTransformer(
    [
        ("scaler", RobustScaler(), _num_cols),
        (
            "ohe",
            OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist"),
            _cat_cols,
        ),
    ]
)

lr_pipe = Pipeline(
    [
        ("preprocesamiento", ct_lr),
        ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
    ]
)

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

print(classification_report(y_test, y_pred_lr))

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=lr_pipe.classes_)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Oranges",
    xticklabels=lr_pipe.classes_,
    yticklabels=lr_pipe.classes_,
    ax=ax,
)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Regresión Logística — Matriz de Confusión (C=1)")
plt.tight_layout()
plt.show()

### Coeficientes del modelo

Una ventaja de la Regresión Logística es que sus **coeficientes son directamente interpretables**: un coeficiente positivo para la clase `k` indica que aumentar esa feature incrementa la probabilidad de que el punto sea de la clase `k`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feat_names = lr_pipe["preprocesamiento"].get_feature_names_out()
coef_df = pd.DataFrame(
    lr_pipe["lr"].coef_,
    index=lr_pipe.classes_,
    columns=feat_names,
).T

fig, ax = plt.subplots(figsize=(9, 5))
coef_df.plot(kind="barh", ax=ax, colormap="tab10")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Coeficientes de Regresión Logística por clase")
ax.set_xlabel("Coeficiente")
plt.tight_layout()
plt.show()

### Widget: Regularización C y Decision Boundary

La regla de oro: **si la frontera de LR tiene mala forma, el problema no es lineal** — no lo resolverás subiendo C, necesitas otro modelo o más feature engineering.

Usa el slider de C (escala logarítmica) para ver el efecto de la regularización y compara la frontera lineal con la del árbol y KNN que viste antes:

In [ ]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler

_feat_opts_lr = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]
_colors_lr = ["steelblue", "darkorange", "green"]


def _show_lr_boundary(log_C, feat_x, feat_y):
    if feat_x == feat_y:
        print("Elige dos features distintas.")
        return

    C = 10**log_C
    X2 = df[[feat_x, feat_y]].values
    y2 = df["species"].values

    scaler2 = RobustScaler()
    X2_sc = scaler2.fit_transform(X2)

    clf_lr2 = LogisticRegression(C=C, max_iter=1000, random_state=42)
    clf_lr2.fit(X2_sc, y2)

    pad = 0.5
    x_min, x_max = X2[:, 0].min() - pad, X2[:, 0].max() + pad
    y_min, y_max = X2[:, 1].min() - pad, X2[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid_sc = scaler2.transform(np.c_[xx.ravel(), yy.ravel()])

    Z = clf_lr2.predict(grid_sc)
    Z_num = np.array([list(clf_lr2.classes_).index(z) for z in Z]).reshape(xx.shape)

    train_acc = clf_lr2.score(X2_sc, y2)
    test_X2 = X_test[[feat_x, feat_y]].values
    test_acc = clf_lr2.score(scaler2.transform(test_X2), y_test)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(
        xx, yy, Z_num, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], colors=_colors_lr
    )
    ax.contour(
        xx, yy, Z_num, levels=[0.5, 1.5], colors="gray", linewidths=0.8, linestyles="--"
    )
    for j, cls in enumerate(clf_lr2.classes_):
        mask = y2 == cls
        ax.scatter(
            X2[mask, 0],
            X2[mask, 1],
            label=cls,
            color=_colors_lr[j],
            edgecolors="k",
            linewidths=0.4,
            s=40,
            zorder=3,
        )
    ax.set_xlabel(feat_x)
    ax.set_ylabel(feat_y)
    ax.set_title(
        f"Reg. Logística — C={C:.3g}   train={train_acc:.2f}  test={test_acc:.2f}"
    )
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


widgets.interact(
    _show_lr_boundary,
    log_C=widgets.FloatSlider(
        min=-3,
        max=3,
        step=0.5,
        value=0,
        description="log₁₀(C):",
        style={"description_width": "initial"},
        readout_format=".1f",
    ),
    feat_x=widgets.Dropdown(
        options=_feat_opts_lr,
        value="flipper_length_mm",
        description="Eje X:",
        style={"description_width": "initial"},
    ),
    feat_y=widgets.Dropdown(
        options=_feat_opts_lr,
        value="culmen_depth_mm",
        description="Eje Y:",
        style={"description_width": "initial"},
    ),
);

> **Pregunta ❓**: ¿La frontera de decisión de la Regresión Logística puede separar perfectamente las tres especies en el espacio 2D? ¿Qué dice eso sobre la separabilidad lineal de este dataset?

### Comparando los tres modelos

Ahora que tenemos Árbol, KNN y Regresión Logística, podemos compararlos directamente. El siguiente widget muestra los tres decision boundaries lado a lado para el mismo par de features, facilitando la comparación visual de sus estilos de frontera.

In [ ]:
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler

_feat_opts_cmp = [
    "culmen_length_mm",
    "culmen_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]
_colors_cmp = ["steelblue", "darkorange", "green"]


def _compare_boundaries(feat_x, feat_y, depth, k, log_C):
    if feat_x == feat_y:
        print("Elige dos features distintas.")
        return

    X2 = df[[feat_x, feat_y]].values
    y2 = df["species"].values

    scaler2 = RobustScaler()
    X2_sc = scaler2.fit_transform(X2)

    models = [
        (
            "Árbol (depth={})".format(depth),
            DecisionTreeClassifier(max_depth=depth, random_state=42),
        ),
        ("KNN (k={})".format(k), KNeighborsClassifier(n_neighbors=k)),
        (
            "Log. Reg. (C={:.2g})".format(10**log_C),
            LogisticRegression(C=10**log_C, max_iter=1000, random_state=42),
        ),
    ]

    pad = 0.5
    x_min, x_max = X2[:, 0].min() - pad, X2[:, 0].max() + pad
    y_min, y_max = X2[:, 1].min() - pad, X2[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid_sc = scaler2.transform(np.c_[xx.ravel(), yy.ravel()])
    grid_raw = np.c_[xx.ravel(), yy.ravel()]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

    for ax, (title, clf) in zip(axes, models):
        # Árbol funciona sobre datos sin escalar (features a la misma escala aquí)
        X_fit = X2_sc if not isinstance(clf, DecisionTreeClassifier) else X2
        X_grid = grid_sc if not isinstance(clf, DecisionTreeClassifier) else grid_raw

        clf.fit(X_fit, y2)
        Z = clf.predict(X_grid)
        Z_num = np.array([list(clf.classes_).index(z) for z in Z]).reshape(xx.shape)
        train_acc = clf.score(X_fit, y2)

        ax.contourf(
            xx, yy, Z_num, alpha=0.25, levels=[-0.5, 0.5, 1.5, 2.5], colors=_colors_cmp
        )
        ax.contour(
            xx,
            yy,
            Z_num,
            levels=[0.5, 1.5],
            colors="gray",
            linewidths=0.8,
            linestyles="--",
        )
        for j, cls in enumerate(clf.classes_):
            mask = y2 == cls
            ax.scatter(
                X2[mask, 0],
                X2[mask, 1],
                label=cls,
                color=_colors_cmp[j],
                edgecolors="k",
                linewidths=0.4,
                s=25,
                zorder=3,
            )
        ax.set_title(f"{title}\ntrain acc={train_acc:.2f}")
        ax.set_xlabel(feat_x)

    axes[0].set_ylabel(feat_y)
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
    plt.suptitle("Comparación de fronteras de decisión", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()


widgets.interact(
    _compare_boundaries,
    feat_x=widgets.Dropdown(
        options=_feat_opts_cmp,
        value="flipper_length_mm",
        description="Eje X:",
        style={"description_width": "initial"},
    ),
    feat_y=widgets.Dropdown(
        options=_feat_opts_cmp,
        value="culmen_depth_mm",
        description="Eje Y:",
        style={"description_width": "initial"},
    ),
    depth=widgets.IntSlider(
        min=1,
        max=10,
        step=1,
        value=3,
        description="Árbol depth:",
        style={"description_width": "initial"},
    ),
    k=widgets.IntSlider(
        min=1,
        max=21,
        step=2,
        value=5,
        description="KNN k:",
        style={"description_width": "initial"},
    ),
    log_C=widgets.FloatSlider(
        min=-2,
        max=2,
        step=0.5,
        value=0,
        description="LR log₁₀(C):",
        style={"description_width": "initial"},
        readout_format=".1f",
    ),
);

> **Pregunta ❓**: Si existiera una herramienta que probara automáticamente decenas de modelos y configuraciones en minutos, ¿cuándo **no** la usarías?

## 🤖 Auto-ML

AutoML (Aprendizaje Automático Automatizado) automatiza las etapas más tediosas del ciclo de ML:

- **Selección del algoritmo**: prueba decenas de modelos de forma sistemática.
- **Optimización de hiperparámetros**: busca la mejor configuración dentro de un presupuesto.
- **Ingeniería de características**: en algunas librerías también aplica transformaciones automáticas.

En lugar de hacer esto manualmente, AutoML lo hace en minutos. Ideal como *baseline* rápido.

### Principales librerías en Python

| Librería | Desarrollador | Características principales |
|---|---|---|
| **FLAML** | Microsoft Research | Muy eficiente en recursos, rápido, fácil de integrar |
| **PyCaret** | Open Source | Interfaz de bajo código, muy visual e interactiva |
| **AutoGluon** | AWS | Muy potente en tabular, imágenes y texto |
| **Auto-sklearn** | Freiburg ML Lab | Basado en sklearn, robusto en benchmarks académicos |

> ⚠️ **Limitación importante**: AutoML no reemplaza entender el problema. Aún necesitas preprocesar datos correctamente, definir métricas adecuadas y validar los resultados.

<div align='center'>
<img src='https://miro.medium.com/v2/1*8lEvj-ilie4HAzMkzkfjqg.png' width=750/>
</div>

<center>FLAML busca el modelo óptimo dentro de un presupuesto de tiempo o recursos.</center>

In [ ]:
# !uv add "flaml[automl]"  # descomentar si no está instalado

from flaml import AutoML

# FLAML acepta DataFrames con columnas categóricas (object/string) directamente.
# No es necesario preprocesar manualmente: FLAML selecciona e incluye
# el preprocesamiento óptimo como parte de la búsqueda del modelo.
automl = AutoML()
automl.fit(
    X_train=X_train,  # DataFrame original, sin transformar
    y_train=y_train,
    task="classification",
    time_budget=60,  # presupuesto en segundos
    metric="accuracy",
    verbose=0,
)

print(f"Mejor estimador : {automl.best_estimator}")
print(f"Accuracy CV est.: {1 - automl.best_loss:.4f}")

In [ ]:
# Evaluamos en el conjunto de test — X_test sin transformar
from sklearn.metrics import classification_report as cr

y_pred_automl = automl.predict(X_test)
print(cr(y_test, y_pred_automl))

---

### 💬 Reflexión: ¿Cuándo *no* usar Auto-ML?

A pesar de su potencia, *Auto-ML* **no reemplaza** entender el problema. Hay situaciones en que no es la mejor elección:

| Situación | Razón |
|---|---|
| **Datos sin limpiar o con *data leakage*** | Auto-ML amplificará los errores del preprocesamiento |
| **Dominio requiere explicabilidad** | Los mejores modelos suelen ser *black boxes* |
| **Recursos computacionales limitados** | Una búsqueda amplia puede tardar horas |
| **Dataset muy pequeño** | El riesgo de *overfitting* es alto sin control del espacio de búsqueda |
| **Requisitos de latencia en producción** | El modelo seleccionado puede ser demasiado pesado |

**Conclusión**: Auto-ML es una herramienta poderosa como *baseline* y para iteraciones rápidas, pero no sustituye el criterio del científico de datos. Úsalo para explorar y comparar, no para reemplazar el entendimiento del problema.